# 🧠 Momentum Optimization and Ravine Dampening

Welcome to the hands-on explanation notebook for **Momentum Optimization**! In this notebook, we will:
1. Explain the physics of Momentum (rolling ball) and Nesterov Accelerated Gradient (look-ahead).
2. Set up a steep 2D ravine cost function:
   $$f(x, y) = 0.5x^2 + 10y^2$$
   where the $y$-axis is 20 times steeper than the $x$-axis.
3. Implement **Vanilla Gradient Descent**, **Momentum GD**, and **Nesterov Accelerated Gradient (NAG)** from scratch.
4. Visualize their trajectories on a 2D contour map to observe how Momentum cancels perpendicular oscillations and accelerates along the valley floor.
5. Compare convergence rates using loss-step curves.
6. Relate these mechanisms to YOLO's default `momentum=0.937` training parameter.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. Defining the Ravine Function and Gradients

Our cost function represents a deep valley:
$$f(x, y) = 0.5x^2 + 10y^2$$

Gradients:
$$\frac{\partial f}{\partial x} = x, \quad \frac{\partial f}{\partial y} = 20y$$

In [ ]:
def cost_ravine(x, y):
    return 0.5 * x**2 + 10.0 * y**2

def grad_ravine(x, y):
    return np.array([x, 20.0 * y])

## 2. Implementing Optimizers from Scratch

Let's implement the three optimization loops:
1.  **Vanilla GD:** $\mathbf{w}_{t+1} = \mathbf{w}_t - \alpha \nabla J(\mathbf{w}_t)$
2.  **Momentum:**
    -   $\mathbf{v}_t = \beta \mathbf{v}_{t-1} + \alpha \nabla J(\mathbf{w}_t)$
    -   $\mathbf{w}_{t+1} = \mathbf{w}_t - \mathbf{v}_t$
3.  **Nesterov Accelerated Gradient (NAG):**
    -   $\mathbf{v}_t = \beta \mathbf{v}_{t-1} + \alpha \nabla J(\mathbf{w}_t - \beta \mathbf{v}_{t-1})$
    -   $\mathbf{w}_{t+1} = \mathbf{w}_t - \mathbf{v}_t$

In [ ]:
def optimize_vanilla(start_pos, lr=0.08, epochs=40):
    pos = np.array(start_pos, dtype=float)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        pos -= lr * grad
        history.append(pos.copy())
    return np.array(history)

def optimize_momentum(start_pos, lr=0.08, beta=0.9, epochs=40):
    pos = np.array(start_pos, dtype=float)
    v = np.zeros(2)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        v = beta * v + lr * grad
        pos -= v
        history.append(pos.copy())
    return np.array(history)

def optimize_nesterov(start_pos, lr=0.08, beta=0.9, epochs=40):
    pos = np.array(start_pos, dtype=float)
    v = np.zeros(2)
    history = [pos.copy()]
    for _ in range(epochs):
        projected_pos = pos - beta * v
        grad = grad_ravine(projected_pos[0], projected_pos[1])
        v = beta * v + lr * grad
        pos -= v
        history.append(pos.copy())
    return np.array(history)

# Run optimizations starting at (8.0, 4.0)
start = [8.0, 4.0]
path_vanilla = optimize_vanilla(start)
path_momentum = optimize_momentum(start)
path_nesterov = optimize_nesterov(start)

## 3. Visualizing Trajectories over the Contour Map

Let's generate the 2D contour grid and plot the paths.

In [ ]:
x = np.linspace(-10, 10, 150)
y = np.linspace(-5, 5, 150)
X, Y = np.meshgrid(x, y)
Z = cost_ravine(X, Y)

plt.figure(figsize=(12, 8))
contours = plt.contour(X, Y, Z, levels=30, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)

# Plot paths
plt.plot(path_vanilla[:, 0], path_vanilla[:, 1], color='red', marker='o', alpha=0.8, linewidth=1.5, label='Vanilla GD (Wild wiggles)')
plt.plot(path_momentum[:, 0], path_momentum[:, 1], color='blue', marker='s', linewidth=2.5, label='Momentum GD (Smooth path)')
plt.plot(path_nesterov[:, 0], path_nesterov[:, 1], color='green', marker='x', linewidth=2, linestyle='--', label='Nesterov GD (Projected lookahead)')

plt.scatter(0, 0, color='gold', s=150, marker='*', zorder=5, label='Minimum (0,0)')
plt.xlabel('x')
plt.ylabel('y')
plt.xlim(-10, 10)
plt.ylim(-5, 5)
plt.title('Dampening Ravine Oscillations: Vanilla vs. Momentum vs. Nesterov')
plt.legend()
plt.show()

Look at the results!
-   **Vanilla GD (Red):** Because the $y$-slope is steep, the updates overshoot, bouncing up and down the walls of the ravine in huge wiggles. It is slow to progress along the $x$-axis.
-   **Momentum (Blue):** Smooths the wiggles! The perpendicular oscillations average out, while consistent speed builds along the $x$-axis.
-   **Nesterov (Green):** Converges even faster with less overshooting due to look-ahead correction braking!

## 4. Comparing Convergence Speeds

Let's plot the cost reduction curves.

In [ ]:
cost_vanilla = [cost_ravine(p[0], p[1]) for p in path_vanilla]
cost_momentum = [cost_ravine(p[0], p[1]) for p in path_momentum]
cost_nesterov = [cost_ravine(p[0], p[1]) for p in path_nesterov]

plt.figure(figsize=(10, 5))
plt.plot(cost_vanilla, color='red', label='Vanilla GD')
plt.plot(cost_momentum, color='blue', label='Momentum GD')
plt.plot(cost_nesterov, color='green', label='Nesterov GD')
plt.yscale('log')
plt.xlabel('Steps')
plt.ylabel('Log Cost')
plt.title('Cost Convergence Comparison (Log Scale)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 💡 Connection to YOLO and Deep Learning
*   **YOLO SGD Momentum:** When training YOLO models, the default momentum factor is set to `0.937` (defined in your YAML training configs). If momentum is too low (e.g. `0.0`), the model takes much longer to train. If momentum is too high (e.g., `0.999`), the model's rolling velocity is so high it will overshoot the optimal weights and diverge.
*   **Look-ahead parameters:** YOLO allows configuring other parameters like weight decay to work alongside momentum to keep weight magnitude stable.